# HAM10000 pipeline

Assumption: the kernel working directory is the project root; `evaluate.py` exposes only a CLI `main()`, so it is run with `%run`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import pandas as pd

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from data import build_cache, config_hash, load_config, load_splits, run_dir
from gradcam import run as gradcam_run
from plots import plot_all
from train import train

from IPython.display import display
from data import run_dir

cfg = load_config("config.json")
print(config_hash(cfg), run_dir(cfg))

In [ ]:
dfs = load_splits(cfg)
build_cache(cfg, "train", dfs["train"])
build_cache(cfg, "val", dfs["val"])  # test cache is built lazily by evaluate.py --final

In [ ]:
RESUME = True  # continues from last.ckpt if present; finished runs are skipped
NAMES = list(cfg["models"])  # edit to train a subset

for name in NAMES:
    train(name, cfg, resume=RESUME)

In [ ]:
%run evaluate.py

In [ ]:
plot_all(cfg, "val")

In [ ]:
for name in [k for k, s in cfg["models"].items() if s["family"] == "transfer"]:
    gradcam_run(name, cfg, split="val")

## Final test evaluation

Reads the test split once, with the pipeline fitted on val (`ensemble.json`). Uncomment only when the val calibration is final.

In [ ]:
%run evaluate.py --final
plot_all(cfg, "test")

In [ ]:

rd = run_dir(cfg)
tdir = os.path.join(rd, "tables")
os.makedirs(tdir, exist_ok=True)
GLOB = ["accuracy", "macro_f1", "balanced_acc", "sev_recall", "auc"]


def _read(p: str) -> dict:
    if not os.path.exists(p):
        return {}
    with open(p) as f:
        return json.load(f)


def _write(text: str, path: str) -> None:
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)


mets = {n: _read(os.path.join(rd, n, "metrics.json")) for n in cfg["models"]}
mets["ensemble"] = _read(os.path.join(rd, "ensemble_metrics.json"))

for split in ("val", "test"):
    rows, pc, sup = {}, {k: {} for k in ("precision", "recall", "f1")}, {}
    for n, m in mets.items():
        if split not in m:
            continue
        r, c = m[split]["raw"], m[split]["calibrated"]
        rows[n] = {
            **{k: c[k] for k in GLOB},
            "ece_raw": r["ece"], "ece_cal": c["ece"],
            "nll_raw": r["nll"], "nll_cal": c["nll"],
        }
        for k in pc:
            pc[k][n] = {cl: c["per_class"][cl][k] for cl in cfg["classes"]}
        sup = {cl: c["per_class"][cl]["support"] for cl in cfg["classes"]}
    if not rows:
        print(f"{split}: no metrics found")
        continue
    tabs = {"global": pd.DataFrame(rows).T}
    tabs.update({f"per_class_{k}": pd.DataFrame(v).T[cfg["classes"]] for k, v in pc.items()})
    tabs["support"] = pd.DataFrame([sup], index=["n"])
    for t, df in tabs.items():
        print(f"{split} | {t}")
        display(df.round(4))
        _write(df.to_csv(float_format="%.4f"), os.path.join(tdir, f"{t}_{split}.csv"))
        try:
            _write(df.to_latex(float_format="%.3f"), os.path.join(tdir, f"{t}_{split}.tex"))
        except Exception as e:
            print(f"latex skipped ({type(e).__name__}): {e}")